In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [0]:
# Date/time range
dbutils.widgets.text("from_date", "17/07/2026", "From Date (dd/MM/yyyy)")
dbutils.widgets.text("from_time", "17:30", "From Time (HH:mm)")
dbutils.widgets.text("to_date", "17/07/2026", "To Date (dd/MM/yyyy)")
dbutils.widgets.text("to_time", "19:15", "To Time (HH:mm)")

REPORT_NAMES = [
    "D.Analysis - OSR PiE",
    "D.Analysis - OSR Topup",
    "D.Analysis - E3 Packing",
    "D.Analysis - Parcel Induct",
    "D.Analysis - Parcel Sortation",
    "D.Analysis - Inbound Decanting",
    "D.Analysis - OSR Decanting",
    "D.Analysis - BCR Inducting",
    "D.Analysis - E1/E2 Inducting",
]

# Filter mode + multi-select of report names
dbutils.widgets.dropdown("filter_mode", "All", ["All", "Include selected", "Exclude selected"], "Filter Mode")
dbutils.widgets.multiselect("selected_reports", REPORT_NAMES[0], REPORT_NAMES, "Select Reports")

from_date = dbutils.widgets.get("from_date")
from_time = dbutils.widgets.get("from_time")
to_date   = dbutils.widgets.get("to_date")
to_time   = dbutils.widgets.get("to_time")
filter_mode = dbutils.widgets.get("filter_mode")
selected = [s.strip() for s in dbutils.widgets.get("selected_reports").split(",") if s.strip()]

print(f"Backfill range: {from_date} {from_time} -> {to_date} {to_time} (UK local time)")
print(f"Filter: {filter_mode}" + (f" -> {selected}" if filter_mode != "All" else ""))

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

uk = ZoneInfo("Europe/London")

start_local = datetime.strptime(f"{from_date} {from_time}", "%d/%m/%Y %H:%M").replace(tzinfo=uk)
end_local   = datetime.strptime(f"{to_date} {to_time}",   "%d/%m/%Y %H:%M").replace(tzinfo=uk)

# Validate alignment to 15-min boundaries
for label, t in [("From", start_local), ("To", end_local)]:
    if t.minute % 15 != 0:
        raise ValueError(f"{label} time {t.strftime('%H:%M')} is not on a 15-minute boundary (:00, :15, :30, :45)")

if end_local <= start_local:
    raise ValueError("To datetime must be after From datetime")

# Build each 15-min block as (start, end) pairs in UK local, plus UTC equivalents for filtering
blocks = []
cursor = start_local
while cursor < end_local:
    block_start_local = cursor
    block_end_local   = cursor + timedelta(minutes=15)
    blocks.append({
        "start_local": block_start_local,
        "end_local":   block_end_local,
        "start_utc":   block_start_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S"),
        "end_utc":     block_end_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S"),
    })
    cursor = block_end_local

print(f"{len(blocks)} blocks to backfill:")
for b in blocks:
    print(f"  {b['start_local'].strftime('%d/%m/%Y %H:%M')} - {b['end_local'].strftime('%d/%m/%Y %H:%M')}")

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"  # adjust to your path

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict, scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1uiv27J0YPDWqSVuN-QLpS-fOcfxAjewK5ajTd3aA8vI"
sh = gc.open_by_key(SHEET_ID)
ws = sh.worksheet("Data")

print("Connected to:", sh.title)

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))
df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

all_reports = [
    {"name": "D.Analysis - OSR PiE",
     "area": "('Pick','Pie Station')", "event": "('COUN','MSKU','PSKU','RSIN')", "extra": ""},
    {"name": "D.Analysis - OSR Topup",
     "area": "('Topup','TopUp','PiOrQi')", "event": "('QSIN','TARS','TPUT')", "extra": ""},
    {"name": "D.Analysis - E3 Packing",
     "area": "('E3 - Packing')", "event": "('PackingParcelCompleteEvent','PackingItemScannedEvent')", "extra": ""},
    {"name": "D.Analysis - Parcel Induct",
     "area": "('Parcel Induct')", "event": "('APAR','PIND','SPAR','UPAR')", "extra": ""},
    {"name": "D.Analysis - Parcel Sortation",
     "area": None, "event": "('ParcelSortedToSack','SackMappedToPosition','SackUnMappedFromPosition')", "extra": ""},
    {"name": "D.Analysis - Inbound Decanting",
     "area": "('Inbound Decanting')", "event": "('DECN','TOPR')", "extra": ""},
    {"name": "D.Analysis - OSR Decanting",
     "area": "('OSR Decanting')", "event": "('ODEC','OTOP')", "extra": ""},
    {"name": "D.Analysis - BCR Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},
    {"name": "D.Analysis - E1/E2 Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND NOT EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},
]

if filter_mode == "Include selected":
    reports = [r for r in all_reports if r['name'] in selected]
elif filter_mode == "Exclude selected":
    reports = [r for r in all_reports if r['name'] not in selected]
else:  # All
    reports = all_reports

if not reports:
    raise ValueError("Your filter selection matched no reports — check Filter Mode and Selected Reports")

print(f"Reports to run ({len(reports)} of {len(all_reports)}):")
for r in reports:
    print(f"  - {r['name']}")

COLUMNS = ['Date', 'Hour', 'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE',
           'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
           'Week', 'Date Time Range', 'Report Name']

In [ ]:
# --- Std Hours divisor (replaces Front!C2). Accepts "96.40%" or a bare number. ---
dbutils.widgets.text("divisor", "96.40%", "Std Hours Divisor")
_div_raw = dbutils.widgets.get("divisor").strip()
divisor = float(_div_raw.rstrip('%')) / 100 if _div_raw.endswith('%') else float(_div_raw)
if divisor == 0:
    divisor = 1.0
print(f"Divisor: {divisor}")

# --- Work-area pivot config: column order for Processed Data (15mins) D:L / N:V,
#     and each area's volume event type(s). Single source of truth. ---
PROC_AREAS = [
    {"report": "D.Analysis - OSR PiE",           "vol_events": {"MSKU", "PSKU"}},
    {"report": "D.Analysis - OSR Topup",         "vol_events": {"TPUT"}},
    {"report": "D.Analysis - E3 Packing",        "vol_events": {"PackingItemScannedEvent"}},
    {"report": "D.Analysis - Parcel Sortation",  "vol_events": {"ParcelSortedToSack"}},
    {"report": "D.Analysis - Parcel Induct",     "vol_events": {"SPAR"}},
    {"report": "D.Analysis - Inbound Decanting", "vol_events": {"DECN"}},
    {"report": "D.Analysis - OSR Decanting",     "vol_events": {"ODEC"}},
    {"report": "D.Analysis - BCR Inducting",     "vol_events": {"SPOS"}},
    {"report": "D.Analysis - E1/E2 Inducting",   "vol_events": {"SPOS"}},
]

In [0]:
import pandas as pd

failures = []
total_rows_pushed = 0

for b in blocks:
    # Per-block metadata, matching the live job's format exactly
    week_val = spark.sql(f"""
      SELECT MOD(FLOOR(DATEDIFF(to_date(from_utc_timestamp(timestamp('{b['start_utc']}'),'Europe/London')), DATE'2025-12-14')/7)+46,52) AS w
    """).collect()[0]['w']

    date_time_range = f"{b['start_local'].strftime('%d/%m/%Y %H:%M')} - {b['end_local'].strftime('%d/%m/%Y %H:%M')}"
    date_display = b['start_local'].strftime('%d/%m/%Y')

    print(f"\n=== Block: {date_time_range} ===")

    for r in reports:
        try:
            area_clause = f"AND PAYLOAD_AREACODE IN {r['area']}" if r['area'] else ""

            query = f"""
              SELECT
                date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'dd/MM/yyyy') AS Date,
                hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London')) AS Hour,
                PAYLOAD_BONUSCODE,
                PAYLOAD_EVENTTYPE,
                SUM(PAYLOAD_QUANTITY)      AS Total_Quantity,
                SUM(PAYLOAD_STANDARDHOURS) AS Total_StandardHours,
                SUM(PAYLOAD_SMV)           AS Total_SMV
              FROM landing_bonus_hub_event_parsed
              WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'X'
                {area_clause}
                AND PAYLOAD_EVENTTYPE IN {r['event']}
                {r['extra']}
                AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{b['start_utc']}')
                AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{b['end_utc']}')
              GROUP BY 1,2,3,4
              ORDER BY Date, Hour, PAYLOAD_BONUSCODE
            """

            df_r = spark.sql(query).toPandas()
            df_r['Week'] = week_val
            df_r['Date Time Range'] = date_time_range
            df_r['Report Name'] = r['name']

            if len(df_r) == 0:
                df_r = pd.DataFrame(
                    [[date_display, '', '', '(no data this window)', '', '', '', week_val, date_time_range, r['name']]],
                    columns=COLUMNS
                )
            else:
                df_r = df_r[COLUMNS]

            values = df_r.astype(str).values.tolist()
            ws.append_rows(values, value_input_option='USER_ENTERED', table_range='A1')
            total_rows_pushed += len(df_r)
            print(f"  [{r['name']}] {len(df_r)} rows")

        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            print(f"  [{r['name']}] FAILED — {error_msg}")
            failures.append({"block": date_time_range, "report": r['name'], "error": error_msg})
            continue

print(f"\n--- Backfill summary ---")
print(f"Blocks processed: {len(blocks)}")
print(f"Total rows pushed: {total_rows_pushed}")
print(f"Failures: {len(failures)}")
for f in failures:
    print(f"  {f['block']} | {f['report']} | {f['error']}")

In [ ]:
# ============================================================
# REBUILD "Processed Data (15mins)" from the Data tab
# Runs once after ALL blocks are appended above, so the rebuild reflects the
# full backfilled range. Pivot: group raw Data rows by (Date Time Range, Hour, Bonus).
#   D:L = sum(StandardHours) per area / divisor ; M = total ;
#   N:V = sum(Quantity) per area for that area's volume event(s).
# Databricks is the sole writer of this tab (Apps Script no longer processes).
# ============================================================
from datetime import datetime
from collections import OrderedDict

def _f(x):
    try:
        return float(str(x).replace(',', '').strip())
    except Exception:
        return 0.0

def _parse_start(dtr):
    try:
        return datetime.strptime(dtr.split(' - ')[0].strip(), '%d/%m/%Y %H:%M')
    except Exception:
        return datetime.max

area_index = {a["report"]: i for i, a in enumerate(PROC_AREAS)}
n_areas = len(PROC_AREAS)

all_data = ws.get_all_values()            # ws = Data worksheet (from earlier cell)
data_rows = all_data[1:] if all_data else []

# Data cols: A=Date B=Hour C=Bonus D=EventType E=Qty F=StdHours G=SMV H=Week I=DateTimeRange J=ReportName
groups = OrderedDict()
for r in data_rows:
    if len(r) < 10:
        continue
    hour = r[1].strip()
    if hour == '':                        # skip "(no data this window)" placeholders
        continue
    ai = area_index.get(r[9].strip())     # Report Name
    if ai is None:
        continue
    key = (r[8].strip(), hour, r[2].strip())   # (Date Time Range, Hour, Bonus)
    g = groups.get(key)
    if g is None:
        g = {"std": [0.0] * n_areas, "vol": [0.0] * n_areas}
        groups[key] = g
    g["std"][ai] += _f(r[5])              # Total_StandardHours (all event types)
    if r[3].strip() in PROC_AREAS[ai]["vol_events"]:
        g["vol"][ai] += _f(r[4])          # Total_Quantity (volume event only)

out = []
for (dtr, hour, bonus), g in groups.items():
    std = [s / divisor for s in g["std"]]
    row = [dtr, hour, bonus] + std + [sum(std)] + g["vol"]
    out.append((_parse_start(dtr), bonus, row))

out.sort(key=lambda t: (t[0], t[1]))
proc_values = [t[2] for t in out]

proc_ws = sh.worksheet("Processed Data (15mins)")
proc_ws.batch_clear(["A2:V"])
if proc_values:
    proc_ws.update("A2", proc_values, value_input_option="USER_ENTERED")

print(f"Processed Data (15mins) rebuilt: {len(proc_values)} rows")